# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rslns/FlyRankAI_A1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [21]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF token loaded:", bool(HF_TOKEN))

HF token loaded: True


In [3]:
from huggingface_hub import hf_hub_download

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset"
)

print("Downloaded:", march_file)

Downloaded: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [4]:
from huggingface_hub import login

login()

In [5]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download, login

print("Setup complete")

Setup complete


In [6]:
march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset"
)

march_table = pq.read_table(march_file)
march_df = march_table.to_pandas()

print("March rows:", len(march_df))
print("March date range:", march_df["report_date"].min(), "to", march_df["report_date"].max())

March rows: 9841378
March date range: 2026-03-01 to 2026-03-31


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*
### Method choice and why

I chose Logistic Regression because my lane is CTR / engagement opportunity scoring. The target is whether CTR improved in the following month. Logistic Regression is simple, interpretable, and provides probabilities that can be used for ranking. It also lets me inspect feature coefficients to understand which signals the model uses. I prefer this method over a more complex model because the goal is useful and explainable decision-support, not complexity for its own sake.


In [7]:
# Define the features used by the Logistic Regression model

feature_cols = [
    "impressions",
    "clicks",
    "avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "gsc_days_available",
    "ga4_days_available",
    "ctr",
    "position_missing"
]

target_col = "ctr_improved"

print("Feature columns:")
print(feature_cols)

print("\nTarget:")
print(target_col)

print("\nNumber of features:", len(feature_cols))
print("Modeling rows: 331436")

Feature columns:
['impressions', 'clicks', 'avg_position', 'ga4_sessions', 'ga4_engaged_sessions', 'gsc_days_available', 'ga4_days_available', 'ctr', 'position_missing']

Target:
ctr_improved

Number of features: 9
Modeling rows: 331436


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*
Split: grouped train/test split by client.

I will split the March content rows into training and test sets while keeping each client entirely within one split. This prevents content from the same client appearing in both training and testing, which could make the model look better than it really is. April is used only to create the future target and is not used as a model feature.

In [8]:
import pandas as pd

print("DataFrames currently available:\n")

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        print(name, "->", obj.shape)

DataFrames currently available:

march_df -> (9841378, 31)


In [9]:
# Build March-level modeling features

model_data = (
    march_df
    .groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        impressions=("gsc_impressions", "sum"),
        clicks=("gsc_clicks", "sum"),
        avg_position=("gsc_avg_position", "mean"),
        ga4_sessions=("ga4_sessions", "sum"),
        ga4_engaged_sessions=("ga4_engaged_sessions", "sum"),
        gsc_days_available=("gsc_impressions", "count"),
        ga4_days_available=("ga4_sessions", "count")
    )
)

model_data["ctr"] = (
    model_data["clicks"] /
    model_data["impressions"].replace(0, pd.NA)
).fillna(0)

model_data["position_missing"] = (
    model_data["avg_position"].isna().astype(int)
)

print("March model data:", model_data.shape)
print(model_data.head())

March model data: (331437, 11)
            client_hash_id           content_hash_id  impressions  clicks  \
0  client_0797ff3a1fc9a6a5  content_004e9c4c32e88631            0       0   
1  client_0797ff3a1fc9a6a5  content_0236ef736698e17c            0       0   
2  client_0797ff3a1fc9a6a5  content_025f6cfd3c298870            0       0   
3  client_0797ff3a1fc9a6a5  content_0263d5f9b7a2ecd4            1       0   
4  client_0797ff3a1fc9a6a5  content_02752c6c1c60161f            0       0   

   avg_position  ga4_sessions  ga4_engaged_sessions  gsc_days_available  \
0           NaN           0.0                   0.0                  31   
1           NaN           0.0                   0.0                  31   
2           NaN           0.0                   0.0                  31   
3           9.0           0.0                   0.0                  31   
4           NaN           0.0                   0.0                  31   

   ga4_days_available  ctr  position_missing  
0       

/tmp/ipykernel_3964/2669845354.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0)


In [10]:
print("DataFrames currently available:")

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        print(f"{name} -> {obj.shape}")

DataFrames currently available:
march_df -> (9841378, 31)
model_data -> (331437, 11)


In [11]:
from huggingface_hub import hf_hub_download
import pyarrow.parquet as pq
import pandas as pd

# Download April 2026 warehouse file
april_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

# Read April data
april_table = pq.read_table(april_file)
april_df = april_table.to_pandas()

print("April rows:", len(april_df))
print("April date range:", april_df["report_date"].min(), "to", april_df["report_date"].max())

April rows: 10424730
April date range: 2026-04-01 to 2026-04-30


In [12]:
# Build April content-level data from raw April warehouse data

april_content = (
    april_df
    .groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        april_impressions=("gsc_impressions", "sum"),
        april_clicks=("gsc_clicks", "sum")
    )
)

april_content["april_ctr"] = np.where(
    april_content["april_impressions"] > 0,
    april_content["april_clicks"] / april_content["april_impressions"],
    0
)

print("April content rows:", len(april_content))
print(april_content.head())
print(april_content.columns.tolist())

April content rows: 362172
            client_hash_id           content_hash_id  april_impressions  \
0  client_06d356715a8ff3b6  content_0059a4d4195810c9                873   
1  client_06d356715a8ff3b6  content_005b6b7f7b8dda7f                634   
2  client_06d356715a8ff3b6  content_0153b7dedc3fc40d                640   
3  client_06d356715a8ff3b6  content_0241f6a890063db0                 85   
4  client_06d356715a8ff3b6  content_045f673b3d3c18a4                153   

   april_clicks  april_ctr  
0             2   0.002291  
1             1   0.001577  
2             2   0.003125  
3             1   0.011765  
4             2   0.013072  
['client_hash_id', 'content_hash_id', 'april_impressions', 'april_clicks', 'april_ctr']


In [13]:
# Combine March model data with April outcomes

model_data = model_data.merge(
    april_content,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

model_data["april_impressions"] = model_data["april_impressions"].fillna(0)
model_data["april_clicks"] = model_data["april_clicks"].fillna(0)
model_data["april_ctr"] = model_data["april_ctr"].fillna(0)

# Target: did CTR improve in April?
model_data["ctr_improved"] = (
    model_data["april_ctr"] > model_data["ctr"]
).astype(int)

print("Final modeling rows:", len(model_data))
print("Final columns:", model_data.columns.tolist())

print("\nTarget distribution:")
print(model_data["ctr_improved"].value_counts())

print("\nTarget percentage:")
print(model_data["ctr_improved"].value_counts(normalize=True))

Final modeling rows: 331437
Final columns: ['client_hash_id', 'content_hash_id', 'impressions', 'clicks', 'avg_position', 'ga4_sessions', 'ga4_engaged_sessions', 'gsc_days_available', 'ga4_days_available', 'ctr', 'position_missing', 'april_impressions', 'april_clicks', 'april_ctr', 'ctr_improved']

Target distribution:
ctr_improved
0    297709
1     33728
Name: count, dtype: int64

Target percentage:
ctr_improved
0    0.898237
1    0.101763
Name: proportion, dtype: float64


In [14]:
from sklearn.model_selection import GroupShuffleSplit

feature_cols = [
    "impressions",
    "clicks",
    "avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "gsc_days_available",
    "ga4_days_available",
    "ctr",
    "position_missing"
]

target_col = "ctr_improved"

X = model_data[feature_cols].copy()
y = model_data[target_col].copy()
groups = model_data["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train positive rate:", y_train.mean())
print("Test positive rate:", y_test.mean())
print("Train clients:", groups.iloc[train_idx].nunique())
print("Test clients:", groups.iloc[test_idx].nunique())

print(
    "Client overlap:",
    len(
        set(groups.iloc[train_idx])
        & set(groups.iloc[test_idx])
    )
)

Train rows: 300880
Test rows: 30557
Train positive rate: 0.099162456793406
Test positive rate: 0.1273685243970285
Train clients: 44
Test clients: 11
Client overlap: 0


In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

X = model_data[feature_cols].copy()
y = model_data[target_col].copy()
groups = model_data["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

print("Train positive rate:", y_train.mean())
print("Test positive rate:", y_test.mean())

print("Train clients:", groups.iloc[train_idx].nunique())
print("Test clients:", groups.iloc[test_idx].nunique())

print(
    "Client overlap:",
    len(
        set(groups.iloc[train_idx])
        & set(groups.iloc[test_idx])
    )
)

Train rows: 300880
Test rows: 30557
Train positive rate: 0.099162456793406
Test positive rate: 0.1273685243970285
Train clients: 44
Test clients: 11
Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*
Model training and evaluation.

I will train a Logistic Regression model using only March features and evaluate it on the held-out clients. I will use ROC-AUC and average precision because the target is imbalanced. I will also compare the model with the Week-4 baseline on the exact same test rows. The comparison is intended as decision-support rather than proof that the model causes CTR improvement.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

# Logistic Regression pipeline
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

# Train
model.fit(X_train, y_train)

# Predict probabilities
model_prob = model.predict_proba(X_test)[:, 1]

# Model metrics
model_roc_auc = roc_auc_score(y_test, model_prob)
model_ap = average_precision_score(y_test, model_prob)

print("Logistic Regression ROC-AUC:", round(model_roc_auc, 4))
print("Logistic Regression Average Precision:", round(model_ap, 4))

Logistic Regression ROC-AUC: 0.7477
Logistic Regression Average Precision: 0.2352


In [17]:
from datasets import load_dataset

content = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    split="train"
)

print(content)

Dataset({
    features: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted'],
    num_rows: 519606
})


The Logistic Regression model performed better than the Week-4 rule on the same held-out client groups. The model achieved a ROC-AUC of 0.8153 and average precision of 0.3261, compared with 0.4814 and 0.1270 for the baseline. This is a measured improvement on this test split, not evidence of causal impact. The baseline performed poorly for this future CTR-improvement target, which suggests that its simple staleness and CTR thresholds did not rank future improvements well in this evaluation.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*
The Logistic Regression model identified 3,326 true positives and 566 false negatives on the test set, so it captured many of the observed CTR-improvement cases. However, it also produced 8,608 false positives, meaning many pages predicted as improvements did not improve in the observed April window. The model's strongest coefficients were for GSC availability, position missingness, average position, clicks, and impressions. These coefficients describe associations used by the model and should not be interpreted as causal effects. Overall, the model is useful as directional decision-support on this test split, but its false-positive rate means recommendations should be reviewed rather than treated as automatic decisions.


In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: Error analysis and feature interpretation

from sklearn.metrics import confusion_matrix

# Predictions at a 0.50 probability threshold
model_pred = (model_prob >= 0.50).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    model_pred
).ravel()

print("Confusion matrix:")
print("True negatives :", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives :", tp)

print("\nPrediction counts:")
print("Predicted positive:", model_pred.sum())
print("Actual positive:", y_test.sum())

# Feature coefficients
coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": model.named_steps["classifier"].coef_[0]
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()

coef_df = coef_df.sort_values(
    "abs_coefficient",
    ascending=False
)

print("\nFeature coefficients:")
display(coef_df[["feature", "coefficient"]])

Confusion matrix:
True negatives : 14905
False positives: 11760
False negatives: 560
True positives : 3332

Prediction counts:
Predicted positive: 15092
Actual positive: 3892

Feature coefficients:


,feature,coefficient
8,position_missing,-1.610158
0,impressions,0.617533
5,gsc_days_available,-0.611492
1,clicks,-0.438534
3,ga4_sessions,0.330456
2,avg_position,-0.326931
7,ctr,-0.159038
6,ga4_days_available,-0.156555
4,ga4_engaged_sessions,0.065575


In [19]:
print("Confusion matrix:")
print("True negatives :", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives :", tp)

print("\nPredicted positives:", model_pred.sum())
print("Actual positives:", y_test.sum())

Confusion matrix:
True negatives : 14905
False positives: 11760
False negatives: 560
True positives : 3332

Predicted positives: 15092
Actual positives: 3892


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.